# 06 · Multi-Agent Orchestration and Events

> **The story:** Tenants email DLD every day asking whether their landlord's rent increase is allowed. The answer depends on Dubai's rules: how far the rent is below the area average, and whether the landlord gave 90 days' notice. Sara, a DLD rental-services officer, gets an AI team to read the email, check the rules, do the maths, draft the reply and book a mediation appointment, and nothing is sent or booked until she approves.

**You will learn**
- A manager agent coordinating the Mail, Rules and Calendar agents
- Three ways of working: you in charge, you give a goal, an email starts it on its own
- Why the team acts as you when you're present, and only drafts when you're not

**Time:** about 25 minutes · **Before you start:**
- Notebooks 00 to 05 finished
- The Work IQ connections from notebook 04

<sub>Workshop material: the rental documents summarise Dubai rules for training only. Not legal advice.</sub>

**Who acts as whom**

| Mode | Who runs the manager | Mailbox and calendar access | Who approves |
|---|---|---|---|
| 1 · You're in charge | this notebook, **as you** | Mail and Calendar agents, as you | you, in this notebook |
| 2 · You give a goal | this notebook, **as you** | Mail and Calendar agents, as you | you, in this notebook |
| 3 · An email starts it | the **hosted manager**, its own identity | none; it only drafts | you, by clicking a button in an email |

In [ ]:
# @title Setup: run this first (about 1 minute, then sign in)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()

settings = WorkshopEnvironment.discover(credential)
print(f"✅ Signed in. Your environment: {settings.resource_group}")

## Mode 1 · You're in charge

Send yourself the tenant email from notebook 04 again if you deleted it. Then give the team one task. The manager plans, delegates and reports; the reply and the booking come back **waiting for your approval**.

In [ ]:
import json
EMAILS = json.loads((ROOT / "workshop/data/tenant_emails.json").read_text())
email = EMAILS["genuine"]
print("Send this email TO YOURSELF from Outlook (copy and paste):\n")
print("Subject:", email["subject"])
print("-" * 60)
print(email["body"])

In [ ]:
from dld_workshop.team import ManagerRun

run = ManagerRun(settings, credential).run("Handle the newest DLD rent email in my inbox.")
table(run.steps, ["step", "tool", "input", "result"])
show("Manager's summary", run.answer)
table(run.pending_table())

### Your decision
Read the proposed reply and appointment above. Set `APPROVE = True` to send the reply and book the appointment (both as you), or leave it `False` to reject them.

In [ ]:
from dld_workshop.agents import decide

APPROVE = False   # ← change to True after reading the proposals
for group in run.pending:
    outcome = decide(settings, credential, group, approve=APPROVE)
    show(f"{group[0]['agent']}: {'approved' if APPROVE else 'rejected'}", outcome["text"])

## Mode 2 · You give a goal

Now send yourself the **second** email: it hides an instruction to forward your whole mailbox. Then give the team a broader goal. Watch the screening step flag it, and notice the forward can't happen anyway: the Mail agent has no forward tool.

In [ ]:
import json
EMAILS = json.loads((ROOT / "workshop/data/tenant_emails.json").read_text())
email = EMAILS["with_hidden_instruction"]
print("Send this email TO YOURSELF from Outlook (copy and paste):\n")
print("Subject:", email["subject"])
print("-" * 60)
print(email["body"])

In [ ]:
goal_run = ManagerRun(settings, credential).run(
    "Prepare a reply for every DLD rent email I received today.")
table(goal_run.steps, ["step", "tool", "input", "result"])
show("Manager's summary", goal_run.answer)
table(goal_run.pending_table())

In [ ]:
APPROVE_GOAL = False   # ← change to True to send the prepared replies
for group in goal_run.pending:
    outcome = decide(settings, credential, group, approve=APPROVE_GOAL)
    show(f"{group[0]['agent']}: {'approved' if APPROVE_GOAL else 'rejected'}", outcome["text"])

## Mode 3 · An email starts it

Nobody runs anything. A Logic App watches your inbox for new emails whose subject starts with **DLD rent**, sends each one to the **hosted manager** (its own identity, no mailbox access), and emails **you** an approval request with three buttons. Only after you click does the Logic App send the reply and book the appointment, through the Outlook connection you authorise.

> ### 👉 Follow along in the portal
>
> **Authorise the Outlook connection (once):** Azure portal → your resource group → the API connection **office365-…** → **Edit API connection** → **Authorize** → sign in → **Save**. The link is printed by the next cell.
> Then open the Logic App **dld-rent-inbox-…** → **Run history** to watch each step live.

In [ ]:
from dld_workshop.events import outlook_connection_status, portal_links

for name, url in portal_links(settings).items():
    print(f"{name}: {url}")
print("\nOutlook connection:", outlook_connection_status(settings, credential))

Once it says **Connected**: send yourself a new email whose subject starts with **DLD rent** (for example *DLD rent question about my renewal*). Within about a minute the Logic App starts; within two, an approval email arrives. Click a button, then run the next cell.

In [ ]:
from dld_workshop.events import logic_app_runs

table(logic_app_runs(settings, credential))

### What just happened
```text
New email in your inbox ─▶ Logic App (polls every minute)
   ─▶ hosted manager (its own identity): screens the email, asks the Rules agent, drafts a reply
   ─▶ approval email to you: [Send reply and book appointment] [Send reply only] [Reject]
   ─▶ on approval: reply sent + appointment booked through your Outlook connection
```

---
## What to notice
- One goal, many steps: the manager decided which agent to call and in what order.
- The hidden instruction was flagged, and the forward was impossible: two layers of defence.
- Every write, in every mode, waited for a person.
- When nobody was signed in, the team could only draft: access follows identity.

**Try it yourself:** in mode 3, choose **Reject** and check that nothing was sent.